# Nano Urdu training on Kaggle (v7)

Import this notebook into Kaggle, enable **GPU** and **Internet**, then run the
cells in order. It trains the 443,402-parameter Nano Urdu voice using
sanoTTS’s `heart-nano` runtime and the `kaab4321/UrduTTS` corpus. This
notebook includes the Nano Urdu Python code and fetches the pinned sanoTTS
source, corpus sample, and training-only aligner.

**v7: GAN-trained decoder (443,402 parameters, unchanged).**
v6 fixed the data (19,796 clips) but still sounded partly robotic. A close-up
spectrogram showed why: even from the true spectrogram, the decoder smeared the
voice harmonics above ~800 Hz, which is the buzzy, metallic sound. It was trained
only with spectral losses, which average away fine harmonic and phase detail.
v7 adds HiFi-GAN-style period discriminators and Vocos-style multi-resolution
spectrogram discriminators with feature matching, the standard way vocoders learn
clean harmonics. The discriminators are used only in training and are not part of
the voice, so the parameter count is unchanged. Decoder training is time-boxed so
the run finishes inside `TIME_LIMIT_HOURS`. Set `DECODER_ADV_WEIGHT = 0` for v6.

Set `SMOKE_TEST = True` for a short pipeline check.
Candidates are filtered during alignment, so the accepted counts can be smaller.
The default corpus sample can contain multiple speakers. Use a curated sample
for a consistent voice. The **Test Urdu script and numbers** cell accepts
Nastaliq Urdu text and plays audio from the trained int8 model.

Download `nano-urdu-pilot-v7.zip` from the notebook output. Use **Save Version → Save & Run All**
to retain outputs after the session ends. [Kaggle notebook guide](https://www.kaggle.com/docs/notebooks)

[MMS-TTS Urdu teacher (CC BY-NC 4.0)](https://huggingface.co/facebook/mms-tts-urd-script_arabic)
 · [Corpus and CC BY-NC 4.0 terms](https://huggingface.co/datasets/kaab4321/UrduTTS)
 · [Pinned sanoTTS source](https://github.com/Ampixa/sanoTTS/tree/55c6f39b2b535b5a591667a0852a6c2778320fab)

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

RUN_START = time.time()

NOTEBOOK_VERSION = "v7 (GAN-trained decoder for clean harmonics)"
print("Nano Urdu notebook", NOTEBOOK_VERSION)
RUN_NAME = "pilot-v7"
# The decoder now trains with discriminators (slower per step). Its two stages
# share whatever time is left before TIME_LIMIT_HOURS, so the run ends in time. Snapshots are saved at
# 1/2 and 3/4 of the acoustic steps in case the session ends early.
SMOKE_TEST = False
REQUIRE_GPU = True
TEACHER = True                  # train on MMS-TTS Urdu teacher audio (consistent voice)
TEACHER_NOISE_SCALE = 0.1       # teacher randomness; lower = more consistent targets (v5: 0.3)
TRAIN_CANDIDATES = 20_000      # sentences (teacher) or recordings (TEACHER = False)
EVAL_CANDIDATES = 300
DURATION_STEPS = 20_000
ACOUSTIC_STEPS = 100_000
DECODER_STEPS = 30_000          # stage 1: decoder on true mels
DECODER_FT_STEPS = 30_000       # stage 2: decoder on predicted mels, int8 simulated
DECODER_PREDICTED_FRACTION = 0.8  # stage 2 batches from acoustic predictions (v5: 0.5)
DECODER_ADV_WEIGHT = 1.0        # GAN discriminators for the decoder (0 = v6 behaviour)
TIME_LIMIT_HOURS = 11.0         # stop decoder training early enough to export and zip
SINGLE_VOICE = True             # recordings only: keep one pitch cluster
ACOUSTIC_ADV_WEIGHT = 0.1       # v3's 0.3 did not help and hurt int8 parity
ACOUSTIC_MSE_WEIGHT = 0.25      # upstream default
MIN_ALIGNMENT_CONFIDENCE = 0.10
# Model sizes (v4: 26 / 31 / 3 = 298,041 parameters). Decoder fixed at 206,122.
DURATION_HIDDEN = 32
ACOUSTIC_HIDDEN = 48
ACOUSTIC_DEPTH = 4             # used for both token and frame depth

CHECKPOINT_STEPS = f"{ACOUSTIC_STEPS // 2},{ACOUSTIC_STEPS * 3 // 4}"  # insurance snapshots
if SMOKE_TEST:
    RUN_NAME = f"{RUN_NAME}-smoke"
    TRAIN_CANDIDATES, EVAL_CANDIDATES = 32, 16
    DURATION_STEPS = ACOUSTIC_STEPS = DECODER_STEPS = DECODER_FT_STEPS = 5
    ACOUSTIC_ADV_WEIGHT = 0.0
    CHECKPOINT_STEPS = ""
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use letters, numbers, underscores, or hyphens in RUN_NAME.")

WORKING = Path("/kaggle/working")
PROJECT = Path("/kaggle/temp/nanourdu")
CACHE = PROJECT / "experiments" / RUN_NAME
RUNS = WORKING / "nano-urdu-output" / RUN_NAME
for directory in (PROJECT, CACHE, RUNS):
    directory.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
ENV.setdefault("OMP_NUM_THREADS", "2")

def run(*args, check=True, stdout=None):
    command = [str(arg) for arg in args]
    print("+ " + " ".join(command), flush=True)
    return subprocess.run(command, cwd=PROJECT, env=ENV, check=check,
                          stdout=stdout, stderr=subprocess.STDOUT if stdout else None)

def python(*args, **kwargs):
    return run(sys.executable, *args, **kwargs)

In [ ]:
# Embedded Nano Urdu source
SOURCES = {
    'Makefile': 'SANOTTS := vendor/sanoTTS\nSANOTTS_REV := 55c6f39b2b535b5a591667a0852a6c2778320fab\nPYTHON ?= python3\nTRAIN_PACK ?= data/train-pack\nEVAL_PACK ?= data/eval-pack\nRUN_DIR ?= runs\nVOICE ?= $(RUN_DIR)/int8\nSTEPS ?= 4000\n\n.PHONY: upstream check-upstream train-front check-voice\nupstream:\n\t@if [ ! -d "$(SANOTTS)/.git" ]; then \\\n\t\tmkdir -p vendor && checkout=$$(mktemp -d vendor/sanoTTS.XXXXXX) && \\\n\t\ttrap \'rm -rf "$$checkout"\' EXIT && \\\n\t\tgit clone --depth 1 --filter=blob:none --sparse --no-checkout https://github.com/Ampixa/sanoTTS.git "$$checkout" && \\\n\t\tgit -C "$$checkout" sparse-checkout set docs tools mcu/src mcu/include mcu/ports/host mcu/test mcu/models/en_us_e13b && \\\n\t\tgit -C "$$checkout" fetch --depth 1 origin "$(SANOTTS_REV)" && \\\n\t\tgit -C "$$checkout" checkout --detach FETCH_HEAD && \\\n\t\ttest -f "$$checkout/mcu/test/fixtures/en_us_e13b/SHA256SUMS" && \\\n\t\tmv "$$checkout" "$(SANOTTS)"; \\\n\tfi\n\t@test "$$(git -C "$(SANOTTS)" rev-parse HEAD)" = "$(SANOTTS_REV)" || \\\n\t\t{ echo "Expected sanoTTS at $(SANOTTS_REV)"; exit 1; }\n\ncheck-upstream: upstream\n\tcd "$(SANOTTS)/mcu/test/fixtures/en_us_e13b" && shasum -a 256 -c SHA256SUMS\n\t$(MAKE) -C "$(SANOTTS)/mcu" test-nano NANO_MODEL=models/en_us_e13b NANO_GOLDEN=test/fixtures/en_us_e13b\n\ntrain-front: upstream\n\t$(PYTHON) audit_budget.py 128 --duration-hidden 32 --acoustic-hidden 48 --acoustic-depth 4\n\t$(PYTHON) "$(SANOTTS)/tools/train_roota_piper_duration_student.py" \\\n\t\t--pack-dir "$(TRAIN_PACK)" --eval-pack-dir "$(EVAL_PACK)" --out-dir "$(RUN_DIR)/duration" \\\n\t\t--hidden 32 --depth 3 --kernel-size 5 --duration-vocab-size 128 \\\n\t\t--max-duration 80 --pause-token-ids "" --steps "$(STEPS)"\n\t$(PYTHON) "$(SANOTTS)/tools/train_roota_piper_latent_student.py" \\\n\t\t--pack-dir "$(TRAIN_PACK)" --eval-pack-dir "$(EVAL_PACK)" --out-dir "$(RUN_DIR)/acoustic" \\\n\t\t--architecture token_context --hidden 48 --depth 4 --token-depth 4 --kernel-size 5 \\\n\t\t--vocab-size 128 --skip-render --steps "$(STEPS)"\n\ncheck-voice: upstream\n\t$(MAKE) -C "$(SANOTTS)/mcu" test-nano NANO_MODEL="$(abspath $(VOICE))" NANO_GOLDEN="$(abspath $(VOICE))/golden"\n',
    'prepare_data.py': '"""Stream a bounded UrduTTS sample for audition and alignment."""\nimport argparse\nimport io\nimport json\nimport time\nfrom math import gcd\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport pyarrow as pa\nimport requests\nfrom scipy.signal import resample_poly\nimport soundfile as sf\n\n\nREPO = "kaab4321/UrduTTS"\nREVISION = "f5c50b7aaabd4325bfe33617232203ad02e30f97"\nBASE = f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}"\nSAMPLE_RATE = 24_000\n\n\ndef download(client, url, path, attempts=8):\n    """Fetch url to path, resuming after dropped connections."""\n    for attempt in range(attempts):\n        done = path.stat().st_size if path.exists() else 0\n        headers = {"Range": f"bytes={done}-"} if done else {}\n        try:\n            with client.get(url, stream=True, timeout=(30, 120), headers=headers) as response:\n                if response.status_code == 416:  # already complete\n                    return\n                response.raise_for_status()\n                mode = "ab" if done and response.status_code == 206 else "wb"\n                with path.open(mode) as stream:\n                    for chunk in response.iter_content(1 << 20):\n                        stream.write(chunk)\n            return\n        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as error:\n            wait = min(60, 5 * 2 ** attempt)\n            print(f"Download interrupted ({error.__class__.__name__}); retrying in {wait}s", flush=True)\n            time.sleep(wait)\n    raise RuntimeError(f"Could not download {url} after {attempts} attempts")\n\n\ndef rows(split):\n    """Yield corpus rows. Each shard is downloaded to disk first, so a slow\n    consumer (e.g. teacher synthesis) never holds a network stream open."""\n    with requests.Session() as client, TemporaryDirectory(prefix="nanourdu-shard-") as temp:\n        response = client.get(f"{BASE}/{split}/state.json", timeout=60)\n        response.raise_for_status()\n        for shard in response.json()["_data_files"]:\n            name = shard["filename"]\n            local = Path(temp) / Path(name).name\n            download(client, f"{BASE}/{split}/{name}", local)\n            try:\n                with pa.memory_map(str(local)) as source, pa.ipc.open_stream(source) as reader:\n                    index = 0\n                    for batch in reader:\n                        audio = batch["audio"]\n                        script = batch["transcription"]\n                        ipa = batch["phonemes"]\n                        for row in range(batch.num_rows):\n                            yield name, index, script[row].as_py(), ipa[row].as_py(), audio[row].as_py()\n                            index += 1\n            finally:\n                local.unlink(missing_ok=True)\n\n\ndef prepare(split, limit, output):\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with TemporaryDirectory(dir=output.parent, prefix=".nanourdu-") as temp:\n        staging = Path(temp) / "sample"\n        (staging / "wavs").mkdir(parents=True)\n        count = 0\n        source = rows(split)\n        try:\n            with (staging / "utterances.jsonl").open("w", encoding="utf-8") as manifest:\n                for shard, index, script, ipa, audio in source:\n                    if not script or not ipa or not audio or not audio["bytes"]:\n                        continue\n                    waveform, rate = sf.read(io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True)\n                    if not 1.5 <= len(waveform) / rate <= 12:\n                        continue\n                    divisor = gcd(rate, SAMPLE_RATE)\n                    mono = resample_poly(waveform.mean(axis=1), SAMPLE_RATE // divisor, rate // divisor)\n                    clip_id = f"{split}-{count:05d}"\n                    relative = f"wavs/{clip_id}.wav"\n                    sf.write(staging / relative, mono.clip(-1, 1), SAMPLE_RATE, subtype="PCM_16")\n                    manifest.write(json.dumps({\n                        "id": clip_id, "audio": relative, "transcription": script, "phonemes": ipa,\n                        "source_file": audio["path"], "source_split": split,\n                        "source_shard": shard, "source_row": index, "source_revision": REVISION,\n                    }, ensure_ascii=False) + "\\n")\n                    count += 1\n                    if count == limit:\n                        break\n        finally:\n            source.close()\n        if count < limit:\n            raise RuntimeError(f"Expected {limit} usable clips and found {count}")\n        staging.rename(output)\n    print(f"Saved {count} clips and their Urdu script and IPA to {output}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", choices=("train", "eval", "test"), default="train")\n    parser.add_argument("--limit", type=int, default=16)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    if args.limit < 1 or args.output.exists():\n        parser.error("provide a positive clip limit and a new output directory")\n    prepare(args.split, args.limit, args.output)\n',
    'align_data.py': '"""Align the corpus IPA with a training-only Wav2Vec2 phoneme aligner."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport re\nimport unicodedata\n\nfrom huggingface_hub import hf_hub_download\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torchaudio.functional import forced_align, resample\nfrom transformers import AutoModelForCTC\n\nfrom build_pack import HOP, SAMPLE_RATE\nfrom phones import BOUNDARY, phone_pattern, tokenize_words, with_boundaries\n\n\nREPO = "facebook/wav2vec2-lv-60-espeak-cv-ft"\nREVISION = "ae45363bf3413b374fecd9dc8bc1df0e24c3b7f4"\nALIGNER_RATE = 16_000\n\n\ndef align(row, audio_path, model, labels, pattern, min_confidence, device="cpu"):\n    words = tokenize_words(row["phonemes"], pattern)\n    phones = [phone for word in words for phone in word]\n    if not 1 <= len(with_boundaries(words)) <= 207:\n        raise ValueError("phone count outside 1–207")\n    audio, rate = sf.read(audio_path, dtype="float32")\n    if audio.ndim != 1 or rate != SAMPLE_RATE:\n        raise ValueError("expected mono 24 kHz audio")\n    waveform = resample(torch.from_numpy(audio), rate, ALIGNER_RATE)\n    waveform = (waveform - waveform.mean()) / (waveform.var(unbiased=False) + 1e-7).sqrt()\n    target = torch.tensor([[labels[phone] for phone in phones]], dtype=torch.int32)\n    with torch.inference_mode():\n        probabilities = model(waveform.unsqueeze(0).to(device)).logits.log_softmax(dim=-1).cpu()\n        if target.numel() + int((target[:, 1:] == target[:, :-1]).sum()) > probabilities.shape[1]:\n            raise ValueError("clip too short for IPA")\n        blank = model.config.pad_token_id\n        path, scores = forced_align(probabilities, target, blank=blank)\n    speech = path[0] != blank\n    confidence = scores[0][speech].exp().mean().item()\n    if confidence < min_confidence:\n        raise ValueError(f"alignment confidence {confidence:.3f} below {min_confidence:.3f}")\n    previous_blank = torch.cat((torch.tensor([True]), ~speech[:-1]))\n    changes = torch.cat((torch.tensor([True]), path[0][1:] != path[0][:-1]))\n    starts = torch.where(speech & (previous_blank | changes))[0].numpy()\n    ends = torch.where(speech & torch.cat((changes[1:], torch.tensor([True]))))[0].numpy()\n    if len(starts) != len(phones) or len(ends) != len(phones):\n        raise ValueError("alignment phone count mismatch")\n\n    stride, receptive_field = 1, 1\n    for kernel, step in zip(model.config.conv_kernel, model.config.conv_stride):\n        receptive_field += (kernel - 1) * stride\n        stride *= step\n    def mel_frame(ctc_frame):\n        seconds = (ctc_frame * stride + (receptive_field - 1) / 2) / ALIGNER_RATE\n        return int(np.rint(seconds * SAMPLE_RATE / HOP))\n\n    total = len(audio) // HOP\n    # Each word gap becomes a boundary token spanning the silence between the\n    # last phone of one word and the first of the next (at least one frame).\n    cuts = [0, max(1, mel_frame(starts[0] - 0.5))]\n    last_of_word = set(np.cumsum([len(word) for word in words])[:-1] - 1)\n    for i in range(len(phones) - 1):\n        if i in last_of_word:\n            left, right = mel_frame(ends[i] + 0.5), mel_frame(starts[i + 1] - 0.5)\n            if right <= left:\n                left = mel_frame((starts[i + 1] + ends[i]) / 2)\n                right = left + 1\n            cuts.extend((left, right))\n        else:\n            cuts.append(mel_frame((starts[i + 1] + ends[i]) / 2))\n    cuts.extend((min(mel_frame(ends[-1] + 0.5), total - 1), total))\n    durations = np.diff(np.asarray(cuts)).astype(int)\n    sequence = with_boundaries(words)\n    if len(durations) != len(sequence):\n        raise ValueError("boundary bookkeeping mismatch")\n    if np.any(durations < 1) or np.any(durations > 80):\n        raise ValueError("duration outside 1–80 mel frames")\n    phones = sequence\n    return {**row, "audio": str(audio_path), "phones": phones, "durations": durations.tolist(),\n            "alignment_confidence": round(confidence, 4)}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("manifest", type=Path)\n    parser.add_argument("output", type=Path)\n    parser.add_argument("vocabulary", type=Path, help="Create this map on train and reuse it for eval.")\n    parser.add_argument("--min-confidence", type=float, default=0.10)\n    parser.add_argument("--local-files-only", action="store_true")\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cpu", help="Device for the Wav2Vec2 forward pass.")\n    args = parser.parse_args()\n    if not 0 <= args.min_confidence <= 1 or args.output.exists():\n        parser.error("choose a confidence in [0, 1] and a new output manifest")\n    pinned = {"revision": REVISION, "local_files_only": args.local_files_only}\n    label_path = hf_hub_download(REPO, "vocab.json", **pinned)\n    raw_labels = json.loads(Path(label_path).read_text(encoding="utf-8"))\n    labels = {unicodedata.normalize("NFC", phone): index for phone, index in raw_labels.items()}\n    pattern = phone_pattern(labels)\n    model = AutoModelForCTC.from_pretrained(REPO, use_safetensors=False, **pinned).eval().to(args.device)\n    existing = args.vocabulary.exists()\n    vocabulary = json.loads(args.vocabulary.read_text(encoding="utf-8")) if existing else {}\n    accepted, observed, fallbacks, skipped = [], set(), 0, {}\n    for line in args.manifest.read_text(encoding="utf-8").splitlines():\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        try:\n            try:\n                aligned = align(row, (args.manifest.parent / row["audio"]).resolve(),\n                                model, labels, pattern, args.min_confidence, args.device)\n            except ValueError as error:\n                # Teacher rows keep the corpus IPA as a fallback when eSpeak emits a\n                # symbol the aligner does not know.\n                if not row.get("phonemes_dataset") or "unknown IPA" not in str(error):\n                    raise\n                row = {**row, "phonemes": row["phonemes_dataset"], "phonemes_source": "dataset"}\n                aligned = align(row, (args.manifest.parent / row["audio"]).resolve(),\n                                model, labels, pattern, args.min_confidence, args.device)\n                fallbacks += 1\n            phones = set(aligned["phones"])\n            if existing and phones - vocabulary.keys():\n                raise ValueError(f"unseen phones {sorted(phones - vocabulary.keys())}")\n        except ValueError as error:\n            print(f"Skipped {row[\'id\']}: {error}")\n            message = str(error)\n            if message.startswith("unknown IPA at "):  # keep the offending symbol\n                reason = "unknown IPA " + message[len("unknown IPA at "):][:3] + "\'"\n            else:\n                reason = re.sub(r"\\d+\\.\\d+", "N", message)\n            skipped[reason] = skipped.get(reason, 0) + 1\n            continue\n        observed.update(phones)\n        accepted.append(aligned)\n    if not accepted:\n        raise SystemExit("No alignments passed")\n    if not existing:\n        if len(observed) > 128:\n            raise SystemExit("The observed inventory exceeds the 128-symbol training configuration")\n        args.vocabulary.parent.mkdir(parents=True, exist_ok=True)\n        with args.vocabulary.open("x", encoding="utf-8") as stream:\n            json.dump({p: i for i, p in enumerate(sorted(observed))}, stream, ensure_ascii=False)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with args.output.open("x", encoding="utf-8") as stream:\n        for row in accepted:\n            stream.write(json.dumps(row, ensure_ascii=False) + "\\n")\n    print(f"Aligned {len(accepted)} clips with {len(observed)} observed symbols "\n          f"({fallbacks} used the corpus IPA fallback)")\n    print("Skip reasons:", json.dumps(dict(sorted(skipped.items(), key=lambda kv: -kv[1])),\n                                      ensure_ascii=False))\n    (args.output.parent / f"{args.output.stem}-skips.json").write_text(\n        json.dumps(skipped, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n',
    'build_pack.py': '"""Convert aligned 24 kHz WAVs into packs the public sanoTTS trainers load."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torchaudio.transforms import MelSpectrogram\n\nSAMPLE_RATE = 24_000\nHOP = 256\nMEL_BANDS = 100\n\n\ndef build(manifest, vocabulary_path, output):\n    vocabulary = json.loads(vocabulary_path.read_text(encoding="utf-8"))\n    if not vocabulary or sorted(vocabulary.values()) != list(range(len(vocabulary))):\n        raise ValueError("Vocabulary must map phone strings to consecutive IDs starting at zero.")\n    mel = MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=HOP,\n                         n_mels=MEL_BANDS, power=1.0, mel_scale="htk")\n    output = output.resolve()\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with TemporaryDirectory(dir=output.parent, prefix=".nanourdu-") as temp:\n        staging = Path(temp) / "pack"\n        staging.mkdir()\n        rows = []\n        with manifest.open(encoding="utf-8") as source, torch.inference_mode():\n            for line in source:\n                if not line.strip():\n                    continue\n                row = json.loads(line)\n                ids = [vocabulary[phone] for phone in row["phones"]]\n                durations = row["durations"]\n                if not ids or len(ids) != len(durations) or any(\n                    type(value) is not int or not 1 <= value <= 80 for value in durations\n                ):\n                    raise ValueError(f"{row[\'id\']}: expected a duration of 1–80 frames per phone")\n                frames = sum(durations)\n                audio, rate = sf.read(manifest.parent / row["audio"], dtype="float32")\n                if rate != SAMPLE_RATE or audio.ndim != 1 or len(audio) < 1024 or len(audio) // HOP != frames:\n                    raise ValueError(f"{row[\'id\']}: expected mono 24 kHz audio matching {frames} frames")\n                waveform = torch.from_numpy(audio[:frames * HOP])\n                target = mel(waveform)[:, :frames].clamp_min(1e-7).log().numpy()\n                filename = f"{len(rows):06d}.npz"\n                np.savez_compressed(staging / filename, phoneme_ids=np.asarray(ids, dtype=np.int64),\n                                    w_ceil=np.asarray(durations, dtype=np.int64), generator_input=target)\n                rows.append({\n                    "row_id": row["id"], "index": len(rows) + 1, "text": row["transcription"],\n                    "chunks": [{"chunk_index": 0, "tensor_npz": str(output / filename),\n                                "audio_samples": frames * HOP}],\n                })\n        if not rows:\n            raise ValueError(f"{manifest}: empty manifest")\n        (staging / "rows.json").write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")\n        staging.rename(output)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("manifest", type=Path)\n    parser.add_argument("vocabulary", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    if args.output.exists():\n        parser.error("choose a new pack output directory")\n    build(args.manifest, args.vocabulary, args.output)\n',
    'audit_budget.py': '"""Count the public sanoTTS front models with a chosen phoneme vocabulary and sizes."""\nimport argparse\nfrom pathlib import Path\nimport sys\n\nTOOLS = Path(__file__).resolve().parent / "vendor/sanoTTS/tools"\nsys.path.insert(0, str(TOOLS))\n\nfrom train_roota_piper_duration_student import DurationStudent\nfrom train_roota_piper_latent_student import ContextualLatentStudent\nfrom decoder import CONFIG, TinyVocosStudent\n\nLIMIT = 500_000\n\n\ndef count(vocabulary, duration_hidden=26, acoustic_hidden=31, acoustic_depth=3):\n    duration = DurationStudent(vocab_size=vocabulary, hidden=duration_hidden, depth=3,\n                               kernel_size=5, max_tokens=207)\n    acoustic = ContextualLatentStudent(vocab_size=vocabulary, hidden=acoustic_hidden,\n                                       depth=acoustic_depth, token_depth=acoustic_depth,\n                                       kernel_size=5, out_channels=100)\n    return tuple(sum(p.numel() for p in model.parameters()) for model in (duration, acoustic))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("vocabulary", type=int)\n    parser.add_argument("--duration-hidden", type=int, default=26)\n    parser.add_argument("--acoustic-hidden", type=int, default=31)\n    parser.add_argument("--acoustic-depth", type=int, default=3)\n    parser.add_argument("--limit", type=int, default=LIMIT)\n    args = parser.parse_args()\n    if not 1 <= args.vocabulary <= 128:\n        parser.error("vocabulary must be between 1 and 128")\n    reference = count(62)\n    if reference != (22_858, 65_299):\n        raise SystemExit(f"Upstream shapes no longer match heart-nano: {reference}")\n    duration, acoustic = count(args.vocabulary, args.duration_hidden,\n                               args.acoustic_hidden, args.acoustic_depth)\n    config = dict(CONFIG)\n    config["dim"] = config.pop("width")\n    decoder = sum(p.numel() for p in TinyVocosStudent(**config).parameters())\n    if decoder != 206_122:\n        raise SystemExit(f"Decoder count does not match heart-nano: {decoder}")\n    total = duration + acoustic + decoder\n    print(f"{args.vocabulary} symbols: {duration:,} duration + {acoustic:,} acoustic + "\n          f"{decoder:,} decoder = {total:,} parameters")\n    if total >= args.limit:\n        raise SystemExit(f"Over the strict under-{args.limit:,} target")\n',
    'decoder.py': '"""Trainable counterpart of the shipped width-62 sanoTTS int8 decoder."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport random\nimport re\nimport sys\nimport time\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nROOT = Path(__file__).resolve().parent\nMCU = ROOT / "vendor/sanoTTS/mcu"\nHOP = 256\nCONFIG = {"n_mels": 100, "width": 62, "num_layers": 4, "expand": 3,\n          "n_fft": 1024, "noise_channels": 4}\n\n\ndef fake_quant(weight):\n    """Per-output-channel symmetric int8 round trip with a straight-through gradient."""\n    scale = weight.abs().amax(dim=tuple(range(1, weight.dim())), keepdim=True).clamp_min(1e-8) / 127\n    return weight + ((weight / scale).round().clamp(-127, 127) * scale - weight).detach()\n\n\ndef conv(layer, x, qat):\n    return F.conv1d(x, fake_quant(layer.weight) if qat else layer.weight, layer.bias,\n                    padding=layer.padding, groups=layer.groups)\n\n\ndef linear(layer, x, qat):\n    return F.linear(x, fake_quant(layer.weight) if qat else layer.weight, layer.bias)\n\n\nclass ConvNeXtBlock(nn.Module):\n    def __init__(self, width, expand):\n        super().__init__()\n        self.dwconv = nn.Conv1d(width, width, 7, padding=3, groups=width)\n        self.norm = nn.LayerNorm(width, eps=1e-6)\n        self.pwconv1 = nn.Linear(width, width * expand)\n        self.pwconv2 = nn.Linear(width * expand, width)\n        self.gamma = nn.Parameter(torch.ones(width))\n\n    def forward(self, x, qat=False):\n        features = self.norm(self.dwconv(x).transpose(1, 2))\n        features = linear(self.pwconv2, F.gelu(linear(self.pwconv1, features, qat)), qat)\n        return x + (features * self.gamma).transpose(1, 2)\n\n\nclass TinyVocosStudent(nn.Module):\n    def __init__(self, *, n_mels, dim, num_layers, expand, n_fft, noise_channels):\n        super().__init__()\n        self.n_fft = n_fft\n        self.embed = nn.Conv1d(n_mels, dim, 7, padding=3)\n        self.noise_adapter = nn.Conv1d(noise_channels, dim, 7, padding=3)\n        self.norm = nn.LayerNorm(dim, eps=1e-6)\n        self.blocks = nn.ModuleList(ConvNeXtBlock(dim, expand) for _ in range(num_layers))\n        self.final_norm = nn.LayerNorm(dim, eps=1e-6)\n        self.head = nn.Linear(dim, n_fft + 2)\n\n    def forward(self, mel, *, noise, qat=False):\n        x = conv(self.embed, mel, qat) + conv(self.noise_adapter, noise, qat)\n        x = self.norm(x.transpose(1, 2)).transpose(1, 2)\n        for block in self.blocks:\n            x = block(x, qat)\n        x = self.final_norm(x.transpose(1, 2))\n        log_magnitude, phase = linear(self.head, x, qat).transpose(1, 2).chunk(2, dim=1)\n        middle = log_magnitude[:, 1:-1].clamp(max=4.605170186).exp()\n        magnitude = F.pad(middle, (0, 0, 1, 1))\n        return torch.polar(magnitude, phase)\n\n    def synthesize(self, spectrum, window):\n        return torch.istft(spectrum, self.n_fft, HOP, window=window)\n\n\ndef load_shipped_decoder():\n    config = dict(CONFIG)\n    config["dim"] = config.pop("width")\n    model = TinyVocosStudent(**config)\n    header = (MCU / "models/en_us_e13b/nano_q8_meta.h").read_text()\n    constants = {k: int(v) for k, v in re.findall(r"^#define (\\w+) (\\d+)$", header, re.M)}\n    blob = (MCU / "test/fixtures/en_us_e13b/model_q8.bin").read_bytes()\n    if len(blob) != constants["NANO_DEC_BYTES"] or sum(p.numel() for p in model.parameters()) != 206_122:\n        raise ValueError("Unexpected heart-nano decoder shape")\n\n    def read(name, count, dtype="<f4"):\n        return np.frombuffer(blob, dtype=dtype, count=count, offset=constants[f"DOFF_{name}"])\n\n    def copy(target, array):\n        target.copy_(torch.from_numpy(np.array(array, dtype=np.float32)).reshape(target.shape))\n\n    def dense(layer, name):\n        out, flat = layer.weight.shape[0], layer.weight[0].numel()\n        width = (flat + 15) // 16 * 16\n        values = read(f"{name}_W8", out * width, "i1").reshape(out, width)[:, :flat]\n        copy(layer.weight, values * read(f"{name}_SCALE", out)[:, None])\n        copy(layer.bias, read(f"{name}_BIAS", out))\n\n    def norm(layer, name):\n        copy(layer.weight, read(f"{name}_W_F32", layer.weight.numel()))\n        copy(layer.bias, read(f"{name}_B_F32", layer.bias.numel()))\n\n    with torch.no_grad():\n        dense(model.embed, "EMBED")\n        dense(model.noise_adapter, "NOISE")\n        norm(model.norm, "NORM")\n        for index, block in enumerate(model.blocks):\n            prefix = f"B{index}"\n            copy(block.dwconv.weight, read(f"{prefix}_DW_W_F32", block.dwconv.weight.numel()))\n            copy(block.dwconv.bias, read(f"{prefix}_DW_B_F32", block.dwconv.bias.numel()))\n            norm(block.norm, f"{prefix}_NORM")\n            dense(block.pwconv1, f"{prefix}_PW0")\n            dense(block.pwconv2, f"{prefix}_PW1")\n            copy(block.gamma, read(f"{prefix}_GAMMA_F32", block.gamma.numel()))\n        norm(model.final_norm, "FNORM")\n        dense(model.head, "HEAD")\n    return model\n\n\ndef load_audio_and_mels(manifest, pack):\n    """Return aligned (log-mel [100, T], waveform) pairs and their row ids."""\n    sources = {row["id"]: manifest.parent / row["audio"] for line in\n               manifest.read_text(encoding="utf-8").splitlines() if line.strip()\n               for row in [json.loads(line)]}\n    rows = json.loads((pack / "rows.json").read_text(encoding="utf-8"))\n    if not rows:\n        raise ValueError("empty pack")\n    items = []\n    for row in rows:\n        with np.load(row["chunks"][0]["tensor_npz"]) as tensors:\n            target = tensors["generator_input"].astype(np.float32)\n        waveform, rate = sf.read(sources[row["row_id"]], dtype="float32")\n        if rate != 24_000 or waveform.ndim != 1 or len(waveform) // HOP != target.shape[-1]:\n            raise ValueError(f"{row[\'row_id\']}: audio does not match its mel target")\n        items.append((row["row_id"], target, waveform[:target.shape[-1] * HOP]))\n    return items\n\n\ndef predicted_mels(pack, checkpoint, device):\n    """Acoustic-model log-mels using the true durations, keyed by row id."""\n    sys.path.insert(0, str(ROOT / "vendor/sanoTTS/tools"))\n    import train_roota_piper_latent_student as acoustic\n    model, _ = acoustic.load_model_from_checkpoint(checkpoint, torch.device(device))\n    model.eval()\n    _, samples, _, _ = acoustic.load_samples(pack)\n    result = {}\n    with torch.no_grad():\n        for sample in samples:\n            features = acoustic.expand_features(sample, torch.device(device))\n            result[sample.row_id] = acoustic.predict_latent_tensor(model, features).T.float().cpu().numpy()\n    return result\n\n\ndef phase_coherence_loss(output, target, n_fft=1024, hop=256, window=None):\n    """Penalise broken phase advance across frames (time) and bins (frequency).\n\n    Incoherent phase is what makes a vocoder sound like several people talking at\n    once. The offset-invariant products below ignore absolute phase, which a mel\n    cannot determine, and only compare how phase moves.\n    """\n    spectra = [torch.stft(x, n_fft, hop, window=window, return_complex=True) for x in (output, target)]\n    loss = 0.0\n    for axis in (-1, -2):\n        parts = []\n        for spectrum in spectra:\n            a, b = (spectrum[..., 1:], spectrum[..., :-1]) if axis == -1 else (spectrum[:, 1:], spectrum[:, :-1])\n            parts.append(a * b.conj())\n        weight = parts[1].abs()\n        unit = [p / (p.abs() + 1e-6) for p in parts]\n        loss = loss + ((1 - (unit[0] * unit[1].conj()).real) * weight).sum() / (weight.sum() + 1e-6)\n    return loss / 2\n\n\ndef spectral_losses(output, audio, windows, resolutions):\n    loss = 0.0\n    for fft in resolutions:\n        a = torch.stft(audio, fft, fft // 4, window=windows[fft], pad_mode="constant", return_complex=True).abs()\n        o = torch.stft(output, fft, fft // 4, window=windows[fft], pad_mode="constant", return_complex=True).abs()\n        loss = loss + (F.l1_loss(o.log1p(), a.log1p()) + F.l1_loss((o + 1e-4).log(), (a + 1e-4).log()) * 0.5\n                       + (a - o).norm() / (a.norm() + 1e-6)) / len(resolutions)\n    return loss\n\n\nclass PeriodDiscriminator(nn.Module):\n    """HiFi-GAN period discriminator: folds the waveform by a period to see pitch-periodic structure."""\n    def __init__(self, period, channels=(32, 64, 128, 256)):\n        super().__init__()\n        self.period = period\n        layers, previous = [], 1\n        for channel in channels:\n            layers.append(nn.utils.parametrizations.weight_norm(\n                nn.Conv2d(previous, channel, (5, 1), (3, 1), padding=(2, 0))))\n            previous = channel\n        layers.append(nn.utils.parametrizations.weight_norm(nn.Conv2d(previous, previous, (5, 1), padding=(2, 0))))\n        self.layers = nn.ModuleList(layers)\n        self.post = nn.utils.parametrizations.weight_norm(nn.Conv2d(previous, 1, (3, 1), padding=(1, 0)))\n\n    def forward(self, audio):\n        x = audio.unsqueeze(1)\n        if x.shape[-1] % self.period:\n            x = F.pad(x, (0, self.period - x.shape[-1] % self.period), mode="reflect")\n        x = x.view(x.shape[0], 1, -1, self.period)\n        features = []\n        for layer in self.layers:\n            x = F.leaky_relu(layer(x), 0.1)\n            features.append(x)\n        return self.post(x).flatten(1), features\n\n\nclass ResolutionDiscriminator(nn.Module):\n    """Vocos-style discriminator on one STFT magnitude resolution: sees harmonic sharpness."""\n    def __init__(self, n_fft, channels=32):\n        super().__init__()\n        self.n_fft = n_fft\n        norm = nn.utils.parametrizations.weight_norm\n        self.layers = nn.ModuleList([\n            norm(nn.Conv2d(1, channels, (3, 9), padding=(1, 4))),\n            norm(nn.Conv2d(channels, channels, (3, 9), stride=(1, 2), padding=(1, 4))),\n            norm(nn.Conv2d(channels, channels, (3, 9), stride=(1, 2), padding=(1, 4))),\n            norm(nn.Conv2d(channels, channels, (3, 9), stride=(1, 2), padding=(1, 4))),\n            norm(nn.Conv2d(channels, channels, (3, 3), padding=(1, 1))),\n        ])\n        self.post = norm(nn.Conv2d(channels, 1, (3, 3), padding=(1, 1)))\n        self.register_buffer("window", torch.hann_window(n_fft), persistent=False)\n\n    def forward(self, audio):\n        spectrum = torch.stft(audio, self.n_fft, self.n_fft // 4, window=self.window, return_complex=True)\n        x = spectrum.abs().clamp_min(1e-5).log().transpose(1, 2).unsqueeze(1)  # [B, 1, frames, bins]\n        features = []\n        for layer in self.layers:\n            x = F.leaky_relu(layer(x), 0.1)\n            features.append(x)\n        return self.post(x).flatten(1), features\n\n\nclass Discriminators(nn.Module):\n    """Training-only critics; they are not part of the exported voice."""\n    def __init__(self):\n        super().__init__()\n        self.members = nn.ModuleList([*(PeriodDiscriminator(p) for p in (2, 3, 5, 7, 11)),\n                                      *(ResolutionDiscriminator(n) for n in (512, 1024, 2048))])\n\n    def forward(self, audio):\n        return [member(audio) for member in self.members]\n\n\ndef discriminator_loss(real, fake):\n    return sum(((1 - r) ** 2).mean() + (f ** 2).mean() for (r, _), (f, _) in zip(real, fake))\n\n\ndef generator_adversarial_loss(real, fake):\n    adversarial = sum(((1 - f) ** 2).mean() for f, _ in fake)\n    matching = sum(F.l1_loss(f, r.detach()) for (_, real_features), (_, fake_features) in zip(real, fake)\n                   for r, f in zip(real_features, fake_features))\n    return adversarial, matching\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("manifest", type=Path)\n    parser.add_argument("pack", type=Path)\n    parser.add_argument("output", type=Path)\n    parser.add_argument("--steps", type=int, default=4000)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--segment-frames", type=int, default=64)\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--min-lr", type=float, default=2e-5)\n    parser.add_argument("--phase-weight", type=float, default=1.0)\n    parser.add_argument("--init", type=Path, help="Continue from a decoder checkpoint instead of heart-nano.")\n    parser.add_argument("--acoustic-checkpoint", type=Path,\n                        help="Also train on this acoustic model\'s predicted mels (what the voice sees at inference).")\n    parser.add_argument("--predicted-fraction", type=float, default=0.5)\n    parser.add_argument("--qat", action="store_true", help="Simulate int8 weights while training.")\n    parser.add_argument("--adv-weight", type=float, default=1.0,\n                        help="GAN loss weight (0 disables the discriminators).")\n    parser.add_argument("--fm-weight", type=float, default=2.0, help="Discriminator feature-matching weight.")\n    parser.add_argument("--recon-weight", type=float, default=15.0,\n                        help="Spectral/phase loss weight relative to the GAN terms.")\n    parser.add_argument("--adv-start", type=int, default=0, help="Step at which the GAN terms switch on.")\n    parser.add_argument("--max-minutes", type=float, default=0,\n                        help="Wall-clock budget; the schedule anneals by steps or time, whichever ends first.")\n    parser.add_argument("--device", choices=("cpu", "cuda"),\n                        default="cuda" if torch.cuda.is_available() else "cpu")\n    args = parser.parse_args()\n    if args.steps < 1 or args.output.exists():\n        parser.error("choose a positive step count and a new output checkpoint")\n    torch.manual_seed(42)\n    rng = np.random.default_rng(42)\n    items = load_audio_and_mels(args.manifest, args.pack)\n    predicted = predicted_mels(args.pack, args.acoustic_checkpoint, args.device) if args.acoustic_checkpoint else {}\n    model = load_shipped_decoder()\n    initial = torch.load(args.init, map_location="cpu") if args.init else {}\n    if initial:\n        model.load_state_dict(initial["model"])\n    model.to(args.device)\n    adversarial = args.adv_weight > 0\n    betas = (0.8, 0.99) if adversarial else (0.9, 0.999)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=betas, weight_decay=1e-4)\n    critics = Discriminators().to(args.device) if adversarial else None\n    if critics is not None and "discriminators" in initial:\n        critics.load_state_dict(initial["discriminators"])  # stage 2 keeps the trained critics\n    critic_optimizer = (torch.optim.AdamW(critics.parameters(), lr=2e-4, betas=(0.8, 0.99))\n                        if critics is not None else None)\n    recon_weight = args.recon_weight if adversarial else 1.0\n    warmup = min(200, args.steps // 10)\n\n    started = time.monotonic()\n\n    def time_progress():\n        return (time.monotonic() - started) / (60 * args.max_minutes) if args.max_minutes > 0 else 0.0\n\n    def learning_rate(step):\n        if step < warmup:\n            return args.lr * (step + 1) / warmup\n        progress = min(1.0, max((step - warmup) / max(1, args.steps - warmup), time_progress()))\n        return args.min_lr + 0.5 * (args.lr - args.min_lr) * (1 + np.cos(np.pi * progress))\n\n    resolutions = (512, 1024, 2048)\n    windows = {n: torch.hann_window(n, device=args.device) for n in resolutions}\n    seg = args.segment_frames\n    for step in range(1, args.steps + 1):\n        if time_progress() >= 1.0:\n            print(f"decoder: time budget of {args.max_minutes:.1f} min reached at step {step - 1}", flush=True)\n            break\n        for group in optimizer.param_groups:\n            group["lr"] = learning_rate(step - 1)\n        if critic_optimizer is not None:\n            for group in critic_optimizer.param_groups:\n                group["lr"] = 2e-4 * learning_rate(step - 1) / args.lr\n        mels, waves = [], []\n        for _ in range(args.batch_size):\n            row_id, mel, wave = items[rng.integers(len(items))]\n            if predicted and rng.random() < args.predicted_fraction:\n                mel = predicted[row_id]\n            start = int(rng.integers(0, max(1, mel.shape[-1] - seg + 1)))\n            mel_crop = mel[:, start:start + seg]\n            wave_crop = wave[start * HOP:(start + mel_crop.shape[-1] - 1) * HOP]\n            if mel_crop.shape[-1] < seg:  # short clip: pad with the quietest mel value and silence\n                pad = seg - mel_crop.shape[-1]\n                mel_crop = np.pad(mel_crop, ((0, 0), (0, pad)), mode="edge")\n                wave_crop = np.pad(wave_crop, (0, (seg - 1) * HOP - len(wave_crop)))\n            mels.append(mel_crop)\n            waves.append(wave_crop)\n        mel = torch.from_numpy(np.stack(mels)).to(args.device)\n        audio = torch.from_numpy(np.stack(waves)).to(args.device)\n        noise = torch.randn(len(mels), CONFIG["noise_channels"], seg, device=args.device)\n        output = model.synthesize(model(mel, noise=noise, qat=args.qat), windows[1024])\n        audio = audio[:, :output.shape[-1]]\n        gan_on = critics is not None and step > args.adv_start\n        # A waveform L1 pushes toward the phase-averaged (buzzy) signal; drop it once the GAN runs.\n        waveform_weight = 0.0 if gan_on else 0.5\n        loss = waveform_weight * F.l1_loss(output, audio) + spectral_losses(output, audio, windows, resolutions)\n        phase = phase_coherence_loss(output, audio, window=windows[1024])\n        total = recon_weight * (loss + args.phase_weight * phase)\n        critic_loss = adv = matching = torch.zeros((), device=args.device)\n        if gan_on:\n            critic_loss = discriminator_loss(critics(audio), critics(output.detach()))\n            critic_optimizer.zero_grad(set_to_none=True)\n            critic_loss.backward()\n            torch.nn.utils.clip_grad_norm_(critics.parameters(), 100.0, error_if_nonfinite=True)\n            critic_optimizer.step()\n            adv, matching = generator_adversarial_loss(critics(audio), critics(output))\n            total = total + args.adv_weight * adv + args.fm_weight * matching\n        optimizer.zero_grad(set_to_none=True)\n        total.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0, error_if_nonfinite=True)\n        optimizer.step()\n        if step % 100 == 0 or step == args.steps or step == 20:\n            print(f"decoder {step}/{args.steps}: loss={loss.item():.4f} phase={phase.item():.4f} "\n                  f"adv={adv.item():.3f} fm={matching.item():.3f} critic={critic_loss.item():.3f} "\n                  f"lr={optimizer.param_groups[0][\'lr\']:.2e}", flush=True)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    checkpoint = {"model": model.cpu().state_dict(), "config": CONFIG}\n    if critics is not None:\n        checkpoint["discriminators"] = critics.cpu().state_dict()\n    torch.save(checkpoint, args.output)\n\n\nif __name__ == "__main__":\n    main()\n',
    'export_nano.py': '"""Run sanoTTS\'s public exporter with the local trainable width-62 decoder."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport shutil\nimport sys\n\nROOT = Path(__file__).resolve().parent\nsys.path.insert(0, str(ROOT / "vendor/sanoTTS/tools"))\nimport export_e12_nano_q8 as exporter\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(add_help=False)\n    parser.add_argument("--out", type=Path)\n    parser.add_argument("--vocabulary", type=Path)\n    options, upstream_args = parser.parse_known_args()\n    if "--help" in upstream_args or "-h" in upstream_args:\n        print("Nano Urdu also requires --vocabulary PHONE_MAP.json")\n    elif not options.vocabulary or not options.out:\n        parser.error("--out and --vocabulary are required")\n    else:\n        if options.out.exists():\n            parser.error("choose a new export directory")\n        vocabulary = json.loads(options.vocabulary.read_text(encoding="utf-8"))\n        if not vocabulary or sorted(vocabulary.values()) != list(range(len(vocabulary))) or len(vocabulary) > 128:\n            parser.error("expected a vocabulary of up to 128 consecutive IDs")\n    sys.argv[1:] = upstream_args\n    if options.out:\n        sys.argv.extend(["--out", str(options.out)])\n    find_upstream_module = exporter._find\n\n    def find_module(*candidates):\n        if all(candidate.name == "train_tiny_vocos_student.py" for candidate in candidates):\n            return find_upstream_module(*candidates, ROOT / "decoder.py")\n        return find_upstream_module(*candidates)\n\n    exporter._find = find_module\n    exporter.main()\n    shutil.copyfile(options.vocabulary, options.out / "phones.json")\n    for blob in options.out.glob("*.bin"):\n        fixture = options.out / "golden" / blob.name\n        if not fixture.exists():\n            fixture.symlink_to(Path("..") / blob.name)\n',
    'speak.py': '"""Render space-separated phonemes with the exported sanoTTS int8 C runtime."""\nimport argparse\nimport ctypes as ct\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nimport soundfile as sf\n\nMCU = Path(__file__).resolve().parent / "vendor/sanoTTS/mcu"\nPCM = ct.CFUNCTYPE(ct.c_int, ct.POINTER(ct.c_float), ct.c_int, ct.c_void_p)\n\n\nclass Config(ct.Structure):\n    _fields_ = [\n        ("front_blob", ct.c_void_p), ("dec_blob", ct.c_void_p), ("arena", ct.c_void_p),\n        ("arena_size", ct.c_size_t), ("dur_override", ct.c_void_p), ("noise_seed", ct.c_uint64),\n    ]\n\n\ndef render(package, ids, seed):\n    report = json.loads((package / "export-report.json").read_text())\n    shapes = report["shapes"]\n    if report["weights"] != "int8":\n        raise ValueError("The host renderer expects an int8 export")\n    if not 2 <= len(ids) <= shapes["NANO_DUR_MAX_TOKENS"]:\n        raise ValueError(f"Expected 2–{shapes[\'NANO_DUR_MAX_TOKENS\']} phonemes for this voice")\n    if any(not 0 <= index < shapes["NANO_VOCAB"] for index in ids):\n        raise ValueError("Phoneme IDs do not match the voice vocabulary")\n    blobs = []\n    for name in ("front_q8.bin", "model_q8.bin"):\n        payload = (package / name).read_bytes()\n        if len(payload) != report["bytes"][name]:\n            raise ValueError(f"Incomplete weight blob {name}")\n        blobs.append(ct.create_string_buffer(payload))\n    # A conservative host scratch bound derived from the maximum mel frames.\n    arena_size = 1024 * 1024 + 1024 * len(ids) * shapes["NANO_DUR_MAX_DURATION"]\n    arena = ct.create_string_buffer(arena_size)\n    config = Config(ct.addressof(blobs[0]), ct.addressof(blobs[1]), ct.addressof(arena),\n                    arena_size, None, seed)\n    samples = []\n\n    @PCM\n    def collect(pcm, length, _user):\n        samples.append(np.ctypeslib.as_array(pcm, (length,)).copy())\n        return 0\n\n    with TemporaryDirectory(prefix="nanourdu-host-") as temp:\n        library = Path(temp) / "sano-nano.so"\n        subprocess.run(["cc", "-shared", "-fPIC", "-O2", "-std=c99", "-DFSD_FAST_MATH",\n                        f"-I{MCU / \'include\'}", f"-I{MCU / \'src\'}", f"-I{package.resolve()}",\n                        str(MCU / "src/snt_nano.c"), str(MCU / "src/snt_kernels_ref.c"),\n                        str(MCU / "ports/host/snt_port_host.c"), "-lm", "-o", str(library)],\n                       check=True)\n        runtime = ct.CDLL(str(library))\n        synthesize = runtime.snt_nano_synthesize\n        synthesize.argtypes = [ct.POINTER(Config), ct.POINTER(ct.c_int32), ct.c_int,\n                               PCM, ct.c_void_p, ct.c_void_p]\n        synthesize.restype = ct.c_int\n        tokens = (ct.c_int32 * len(ids))(*ids)\n        status = synthesize(ct.byref(config), tokens, len(ids), collect, None, None)\n    if status != 0 or not samples:\n        raise RuntimeError(f"sanoTTS synthesis returned {status}")\n    return np.concatenate(samples)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("package", type=Path)\n    parser.add_argument("phones", help="One argument with a space between each phoneme.")\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    vocabulary = json.loads((args.package / "phones.json").read_text(encoding="utf-8"))\n    try:\n        ids = [vocabulary[phone] for phone in args.phones.split()]\n    except KeyError as error:\n        parser.error(f"phoneme {error.args[0]!r} is missing from this voice")\n    seed = int.from_bytes(hashlib.sha256(args.phones.encode()).digest()[:8], "big")\n    waveform = render(args.package, ids, seed)\n    with args.output.open("xb") as output:\n        sf.write(output, waveform, 24_000, format="WAV", subtype="PCM_16")\n    print(f"Wrote {len(waveform) / 24_000:.2f} s to {args.output}")\n',
    'phones.py': '"""Shared IPA segmentation for corpus alignment and synthesis."""\nimport re\nimport unicodedata\n\nBOUNDARY = "|"  # word-boundary / pause token; carries the silence between words\nLENGTH = "ː"\n\n\ndef phone_pattern(inventory):\n    symbols = sorted((phone for phone in inventory\n                      if phone and not phone.startswith("<") and phone != BOUNDARY),\n                     key=lambda phone: (-len(phone), phone))\n    if not symbols:\n        raise ValueError("The phone inventory is empty.")\n    return re.compile("|".join(map(re.escape, symbols)))\n\n\ndef tokenize_words(ipa, pattern):\n    """IPA split into words, each a list of phones."""\n    ipa = unicodedata.normalize("NFC", ipa).translate(str.maketrans("", "", "ˈˌ͜͡\u200c\u200d"))\n    words = []\n    for word in re.split(r"[\\s.,:;!?،؛؟۔…“”|]+", ipa):\n        phones, offset = [], 0\n        while offset < len(word):\n            match = pattern.match(word, offset)\n            if match is None:\n                symbol = word[offset]\n                if symbol == LENGTH and phones:\n                    # A length mark the inventory has no combined symbol for\n                    # (eSpeak writes Urdu geminates as e.g. "nː"): hold the\n                    # previous phone for twice as long by repeating it.\n                    phones.append(phones[-1])\n                    offset += 1\n                    continue\n                if unicodedata.combining(symbol):\n                    offset += 1  # an unsupported diacritic (e.g. stray nasal tilde)\n                    continue\n                raise ValueError(f"unknown IPA at {word[offset:]!r}")\n            phones.append(match[0])\n            offset = match.end()\n        if phones:\n            words.append(phones)\n    return words\n\n\ndef tokenize(ipa, pattern):\n    return [phone for word in tokenize_words(ipa, pattern) for phone in word]\n\n\ndef with_boundaries(words):\n    """Phones with a boundary token before, between, and after words."""\n    sequence = [BOUNDARY]\n    for word in words:\n        sequence.extend(word)\n        sequence.append(BOUNDARY)\n    return sequence\n',
    'urdu_frontend.py': '"""Urdu-script text and number normalization for Nano Urdu."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport unicodedata\n\nfrom phones import BOUNDARY, LENGTH, phone_pattern, tokenize_words, with_boundaries\n\n\nCARDINALS = """\nصفر ایک دو تین چار پانچ چھ سات آٹھ نو\nدس گیارہ بارہ تیرہ چودہ پندرہ سولہ سترہ اٹھارہ انیس\nبیس اکیس بائیس تئیس چوبیس پچیس چھبیس ستائیس اٹھائیس انتیس\nتیس اکتیس بتیس تینتیس چونتیس پینتیس چھتیس سینتیس اڑتیس انتالیس\nچالیس اکتالیس بیالیس تینتالیس چوالیس پینتالیس چھیالیس سینتالیس اڑتالیس انچاس\nپچاس اکیاون باون ترپن چوّن پچپن چھپن ستاون اٹھاون انسٹھ\nساٹھ اکسٹھ باسٹھ تریسٹھ چونسٹھ پینسٹھ چھیاسٹھ سڑسٹھ اڑسٹھ انہتر\nستر اکہتر بہتر تہتر چوہتر پچہتر چھہتر ستتر اٹھہتر اناسی\nاسی اکیاسی بیاسی تراسی چوراسی پچاسی چھیاسی ستاسی اٹھاسی نواسی\nنوے اکانوے بانوے ترانوے چورانوے پچانوے چھیانوے ستانوے اٹھانوے ننانوے\n""".split()\nSCALES = ((10**12, "کھرب"), (10**9, "ارب"), (10**7, "کروڑ"),\n          (10**5, "لاکھ"), (1000, "ہزار"), (100, "سو"))\nDIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩كي−٫٬٪",\n                          "01234567890123456789کی-.,%")\nNUMBER = re.compile(r"[+-]?[0-9]+(?:,[0-9]+)*(?:\\.[0-9]+)?(?:[ \\t]*%)?")\nGROUPING = re.compile(r"(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]{1,2}(?:,[0-9]{2})+,[0-9]{3})")\nPUNCTUATION = "۔،؛؟!?.,:;…()«»\\"\'“”–—-"\n\n\ndef cardinal(number):\n    words = []\n    for scale, name in SCALES:\n        units, number = divmod(number, scale)\n        if units:\n            words.extend((CARDINALS[units], name))\n    if number or not words:\n        words.append(CARDINALS[number])\n    return " ".join(words)\n\n\ndef spell_digits(number):\n    return " ".join(CARDINALS[int(digit)] for digit in number)\n\n\ndef speak_number(match, digit_by_digit):\n    token = match[0]\n    sign = {"-": "منفی", "+": "مثبت"}.get(token[0])\n    percent = token.endswith("%")\n    unsigned = token.lstrip("+-").removesuffix("%").strip()\n    integer, _, fraction = unsigned.partition(".")\n    if "," in integer and not GROUPING.fullmatch(integer):\n        raise ValueError(f"Invalid digit grouping in {token!r}. Use spaces between separate numbers.")\n    integer = integer.replace(",", "")\n    if digit_by_digit or len(integer) > 14 or (len(integer) > 1 and integer.startswith("0")):\n        words = [spell_digits(integer)]\n    else:\n        words = [cardinal(int(integer))]\n    if sign:\n        words.insert(0, sign)\n    if fraction:\n        words.extend(("اعشاریہ", spell_digits(fraction)))\n    if percent:\n        words.append("فیصد")\n    return " " + " ".join(words) + " "\n\n\ndef normalize(text, *, digit_by_digit=False):\n    text = unicodedata.normalize("NFKC", text).translate(DIGIT_MAP)\n    text = "".join(char for char in text if unicodedata.category(char) != "Cf" and char != "ـ")\n    text = re.sub(r"[0-9]+(?:،[0-9]+)+", normalize_grouping, text)\n    if re.search(r"[0-9]\\s*[:/–—-]\\s*[0-9]", text):\n        raise ValueError("Write times, fractions, dates, and ranges out in Urdu words.")\n    text = NUMBER.sub(lambda match: speak_number(match, digit_by_digit), text).strip()\n    for char in text:\n        if char.isspace() or char in PUNCTUATION:\n            continue\n        kind = unicodedata.category(char)[0]\n        if kind not in "LM" or not unicodedata.name(char, "").startswith("ARABIC"):\n            raise ValueError(f"Use Urdu script and numbers. Unsupported character {char!r}.")\n    if not any(unicodedata.category(char).startswith("L") for char in text):\n        raise ValueError("Enter some Urdu text or a number.")\n    return re.sub(r"\\s+", " ", text)\n\n\ndef normalize_grouping(match):\n    grouped = match[0].replace("،", ",")\n    return grouped if GROUPING.fullmatch(grouped) else match[0]\n\n\ndef phonemize(text):\n    executable = shutil.which("espeak-ng")\n    if executable is None:\n        raise RuntimeError("Install eSpeak NG for Urdu phonemization.")\n    result = subprocess.run([executable, "-q", "--ipa", "-v", "ur", "--stdin"],\n                            input=text + "\\n", text=True, capture_output=True, check=True)\n    return result.stdout.strip()\n\n\nVOWELS = "aeiouɛɔəɪʊʌɐ"\n# Nearest sounds for symbols eSpeak can emit (mostly in loanwords) that a voice\n# may never have seen in training; used only when the voice lacks the symbol.\nNEAREST = {"ʒ": "z", "g": "ɡ", "ŋ": "n", "ɲ": "n", "ɳ": "n", "ɭ": "l", "v": "ʋ",\n           "ɦ": "h", "χ": "x", "ʁ": "ɣ", "ɑ": "aː", "ɑː": "aː", "æ": "ɛ", "ɒ": "ɔ"}\n\n\ndef phone_aliases(vocabulary):\n    """Map phones the voice never saw to the nearest one it knows (e.g. loanword "o" -> "oː")."""\n    aliases = {source: target for source, target in NEAREST.items()\n               if source not in vocabulary and target in vocabulary}\n    for vowel in VOWELS:\n        for base in (vowel, vowel + "̃"):  # plain and nasal\n            short, long = unicodedata.normalize("NFC", base), unicodedata.normalize("NFC", base + LENGTH)\n            if short not in vocabulary and long in vocabulary:\n                aliases[short] = long\n            elif long not in vocabulary and short in vocabulary:\n                aliases[long] = short\n    return aliases\n\n\ndef phones_for(ipa, vocabulary):\n    aliases = phone_aliases(vocabulary)\n    words = tokenize_words(ipa, phone_pattern(set(vocabulary) | set(aliases)))\n    words = [[aliases.get(phone, phone) for phone in word] for word in words]\n    if not words:\n        raise ValueError("The phonemizer produced no usable phones.")\n    if BOUNDARY in vocabulary:  # voices trained with word-boundary / pause tokens\n        return with_boundaries(words)\n    return [phone for word in words for phone in word]\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("package", type=Path)\n    parser.add_argument("text")\n    parser.add_argument("output", type=Path)\n    parser.add_argument("--digits", action="store_true", help="Read every integer digit individually.")\n    args = parser.parse_args()\n    try:\n        spoken = normalize(args.text, digit_by_digit=args.digits)\n        vocabulary = json.loads((args.package / "phones.json").read_text(encoding="utf-8"))\n        ipa = phonemize(spoken)\n        phones = phones_for(ipa, vocabulary)\n    except (ValueError, RuntimeError, subprocess.CalledProcessError) as error:\n        parser.error(str(error))\n    print(f"Spoken Urdu: {spoken}\\nIPA: {ipa}", flush=True)\n    subprocess.run([sys.executable, str(Path(__file__).with_name("speak.py")),\n                    str(args.package), " ".join(phones), str(args.output)], check=True)\n',
    'speaker_filter.py': '"""Keep the clips of one voice, chosen by median pitch (UrduTTS has no speaker labels).\n\nTraining one model on a mix of speakers makes it predict an average of their pitches,\nwhich sounds like several people talking at once. This keeps the most common pitch\ncluster. Evaluation clips reuse the training cluster via --center.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nimport torchaudio.functional as AF\n\nRATE = 24_000\nHOP = 240  # 10 ms\n\n\ndef pitch_track(path):\n    wave, rate = sf.read(path, dtype="float32")\n    if rate != RATE or wave.ndim != 1:\n        raise ValueError(f"{path}: expected mono {RATE} Hz audio")\n    wave = torch.from_numpy(wave)\n    pitch = AF.detect_pitch_frequency(wave, RATE, frame_time=HOP / RATE, freq_low=70, freq_high=400)\n    frames = min(len(pitch), len(wave) // HOP)\n    energy = wave[:frames * HOP].reshape(frames, HOP).pow(2).mean(1).sqrt()\n    voiced = (energy > 0.25 * energy.max()) & (pitch[:frames] > 75) & (pitch[:frames] < 390)\n    if voiced.sum() < 20:\n        return None\n    return pitch[:frames][voiced].numpy()\n\n\ndef median_pitch(path):\n    track = pitch_track(path)\n    return None if track is None else float(np.median(track))\n\n\ndef pitch_variation(path):\n    """Standard deviation of the voiced pitch contour in semitones (flat, robotic speech is low)."""\n    track = pitch_track(path)\n    return None if track is None else float(np.std(12 * np.log2(track / np.median(track))))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("manifest", type=Path)\n    parser.add_argument("output", type=Path, help="Filtered manifest, written beside the audio.")\n    parser.add_argument("--center", type=float, help="Pitch in Hz to keep; default: the most common one.")\n    parser.add_argument("--tolerance", type=float, default=0.10, help="Allowed fraction around the centre.")\n    parser.add_argument("--min-clips", type=int, default=20)\n    args = parser.parse_args()\n    if args.output.exists() or args.output.parent != args.manifest.parent:\n        parser.error("choose a new output manifest in the same directory as the input")\n    rows = [json.loads(line) for line in args.manifest.read_text(encoding="utf-8").splitlines() if line.strip()]\n    pitches = [median_pitch(args.manifest.parent / row["audio"]) for row in rows]\n    valid = np.array([p for p in pitches if p])\n    center = args.center\n    if center is None:\n        grid = np.arange(80, 380, 2.0)\n        score = [(np.abs(np.log(valid / g)) < args.tolerance).sum() for g in grid]\n        center = float(grid[int(np.argmax(score))])\n    keep = [row for row, p in zip(rows, pitches) if p and abs(np.log(p / center)) < args.tolerance]\n    histogram, edges = np.histogram(valid, bins=np.arange(70, 400, 20))\n    print("median-pitch histogram (Hz: clips)")\n    for count, low in zip(histogram, edges):\n        print(f"  {int(low):3d}-{int(low) + 19:3d}: {\'#\' * int(40 * count / max(1, histogram.max()))} {count}")\n    print(f"keeping {len(keep)} of {len(rows)} clips within {args.tolerance:.0%} of {center:.0f} Hz")\n    if len(keep) < args.min_clips:\n        raise RuntimeError("Too few clips in one pitch cluster; raise the candidate count or --tolerance.")\n    with args.output.open("w", encoding="utf-8") as out:\n        for row in keep:\n            out.write(json.dumps(row, ensure_ascii=False) + "\\n")\n    args.output.with_suffix(".json").write_text(json.dumps(\n        {"center_hz": center, "tolerance": args.tolerance, "kept": len(keep), "total": len(rows)}))\n\n\nif __name__ == "__main__":\n    main()\n',
    'compare_voice.py': '"""Render held-out clips through each stage so you can hear where quality is lost.\n\nFor every clip it writes:\n  1_ground_truth.wav   the original recording\n  2_decoder_only.wav   true mel -> decoder        (if this is bad, the decoder is the problem)\n  3_acoustic.wav       predicted mel -> decoder   (if only this is bad, the acoustic model is)\n  4_int8_runtime.wav   phonemes -> int8 C runtime (what ships; if only this is bad, it is quantization)\nplus mels.png (true vs predicted log-mel) and report.json.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\n\nimport decoder as dec\nimport speaker_filter\nfrom speak import render\n\nRATE = 24_000\n\n\ndef load_decoder(path, device):\n    model = dec.load_shipped_decoder()\n    model.load_state_dict(torch.load(path, map_location="cpu")["model"])\n    return model.to(device).eval()\n\n\ndef vocode(model, mel, device, seed):\n    generator = torch.Generator(device="cpu").manual_seed(seed)\n    noise = torch.randn(1, dec.CONFIG["noise_channels"], mel.shape[-1], generator=generator).to(device)\n    window = torch.hann_window(dec.CONFIG["n_fft"], device=device)\n    with torch.no_grad():\n        return model.synthesize(model(torch.from_numpy(mel)[None].to(device), noise=noise), window)[0].cpu().numpy()\n\n\ndef write(path, wave):\n    sf.write(path, np.clip(wave, -1, 1), RATE, subtype="PCM_16")\n    return path\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("manifest", type=Path, help="aligned.jsonl of the evaluation split")\n    parser.add_argument("pack", type=Path)\n    parser.add_argument("--decoder", type=Path, required=True)\n    parser.add_argument("--acoustic", type=Path, required=True)\n    parser.add_argument("--voice", type=Path, required=True, help="int8 export directory")\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--count", type=int, default=4)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    args = parser.parse_args()\n    args.out.mkdir(parents=True, exist_ok=True)\n    items = dec.load_audio_and_mels(args.manifest, args.pack)[:args.count]\n    predicted = dec.predicted_mels(args.pack, args.acoustic, args.device)\n    rows = {row["row_id"]: row for row in json.loads((args.pack / "rows.json").read_text(encoding="utf-8"))}\n    model = load_decoder(args.decoder, args.device)\n    report = []\n    figure_rows = []\n    for index, (row_id, mel, wave) in enumerate(items):\n        folder = args.out / f"{index:02d}-{row_id}"\n        folder.mkdir(exist_ok=True)\n        pred = predicted[row_id]\n        with np.load(rows[row_id]["chunks"][0]["tensor_npz"]) as tensors:\n            ids = [int(i) for i in tensors["phoneme_ids"]]\n        files = {"1_ground_truth": write(folder / "1_ground_truth.wav", wave),\n                 "2_decoder_only": write(folder / "2_decoder_only.wav", vocode(model, mel, args.device, 1)),\n                 "3_acoustic": write(folder / "3_acoustic.wav", vocode(model, pred, args.device, 1))}\n        try:\n            files["4_int8_runtime"] = write(folder / "4_int8_runtime.wav", render(args.voice, ids, 1))\n        except Exception as error:  # keep going: the float stages are still informative\n            print(f"{row_id}: int8 runtime failed: {error}")\n        a, b = mel.reshape(-1), pred.reshape(-1)\n        report.append({\n            "row": row_id, "text": rows[row_id]["text"], "files": {k: str(v) for k, v in files.items()},\n            "mel_l1": float(np.abs(mel - pred).mean()),\n            "mel_cosine": float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))),\n            "median_pitch_hz": {k: speaker_filter.median_pitch(v) for k, v in files.items()},\n            "pitch_variation_semitones": {k: speaker_filter.pitch_variation(v) for k, v in files.items()},\n        })\n        figure_rows.append((row_id, mel, pred))\n    try:\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n        fig, axes = plt.subplots(len(figure_rows), 2, figsize=(12, 2.4 * len(figure_rows)), squeeze=False)\n        for axis, (row_id, mel, pred) in zip(axes, figure_rows):\n            low, high = np.percentile(mel, [1, 99])\n            axis[0].imshow(mel, origin="lower", aspect="auto", vmin=low, vmax=high)\n            axis[0].set_title(f"{row_id}: true log-mel")\n            axis[1].imshow(pred, origin="lower", aspect="auto", vmin=low, vmax=high)\n            axis[1].set_title("predicted log-mel (blurry = chorus effect)")\n        fig.tight_layout()\n        fig.savefig(args.out / "mels.png", dpi=90)\n    except ImportError:\n        pass\n    (args.out / "report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))\n    print(json.dumps([{k: v for k, v in r.items() if k not in ("files", "text")} for r in report], indent=2))\n\n\nif __name__ == "__main__":\n    main()\n',
    'teacher_data.py': '"""Build a consistent one-voice training corpus by reading UrduTTS sentences with a teacher TTS.\n\nWhy: a 300k-parameter model trained on human recordings learns the *average* of\nmany different ways each phrase was spoken, which sounds flat and robotic. A\nteacher model (Meta MMS-TTS Urdu, a VITS model) speaks every phrase one fixed way,\nso the small student can learn that mapping instead of an average. The student\'s\nvoice and prosody then follow the teacher.\n\nEach row gets the teacher audio at 24 kHz plus IPA from the same eSpeak NG front\nend that synthesis uses, so training and inference see identical phones.\n"""\nimport argparse\nimport json\nfrom math import gcd\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nfrom scipy.signal import resample_poly\nimport soundfile as sf\nimport torch\n\nfrom prepare_data import rows\nfrom urdu_frontend import normalize, phonemize\n\nTEACHER = "facebook/mms-tts-urd-script_arabic"\nTEACHER_REVISION = "4c4f53d52e4f0b40cb8e7f36dcef66a2a73619c5"\nSAMPLE_RATE = 24_000\n\n\ndef trim_silence(wave, rate, threshold_db=-40, pad_seconds=0.08):\n    frame = int(rate * 0.01)\n    energy = np.sqrt(np.convolve(wave ** 2, np.ones(frame) / frame, mode="same"))\n    loud = np.flatnonzero(energy > energy.max() * 10 ** (threshold_db / 20))\n    if not len(loud):\n        return wave\n    pad = int(rate * pad_seconds)\n    return wave[max(0, loud[0] - pad):loud[-1] + pad]\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--split", choices=("train", "eval", "test"), default="train")\n    parser.add_argument("--limit", type=int, default=16)\n    parser.add_argument("--skip", type=int, default=0, help="Skip this many usable sentences first.")\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--noise-scale", type=float, default=0.3,\n                        help="Teacher prosody randomness. Lower is more consistent (easier to learn).")\n    parser.add_argument("--noise-scale-duration", type=float, default=0.0)\n    parser.add_argument("--phonemizer", choices=("espeak", "dataset"), default="espeak",\n                        help="espeak matches synthesis; dataset uses the corpus IPA column.")\n    args = parser.parse_args()\n    if args.limit < 1 or args.output.exists():\n        parser.error("provide a positive limit and a new output directory")\n\n    from transformers import AutoTokenizer, VitsModel\n    tokenizer = AutoTokenizer.from_pretrained(TEACHER, revision=TEACHER_REVISION)\n    model = VitsModel.from_pretrained(TEACHER, revision=TEACHER_REVISION).to(args.device).eval()\n    model.noise_scale = args.noise_scale\n    model.noise_scale_duration = args.noise_scale_duration\n    teacher_rate = model.config.sampling_rate\n    divisor = gcd(teacher_rate, SAMPLE_RATE)\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    with TemporaryDirectory(dir=args.output.parent, prefix=".nanourdu-") as temp:\n        staging = Path(temp) / "teacher"\n        (staging / "wavs").mkdir(parents=True)\n        count = skipped = seen = 0\n        source = rows(args.split)\n        failure = None\n        try:\n          try:\n            with (staging / "utterances.jsonl").open("w", encoding="utf-8") as manifest:\n                for shard, index, script, dataset_ipa, _audio in source:\n                    try:\n                        text = normalize(script or "")\n                    except ValueError:\n                        skipped += 1\n                        continue\n                    seen += 1\n                    if seen <= args.skip:\n                        continue\n                    ipa = phonemize(text) if args.phonemizer == "espeak" else dataset_ipa\n                    inputs = tokenizer(text, return_tensors="pt").to(args.device)\n                    if not ipa or inputs["input_ids"].shape[-1] < 3:\n                        skipped += 1\n                        continue\n                    torch.manual_seed(count)\n                    with torch.no_grad():\n                        wave = model(**inputs).waveform[0].float().cpu().numpy()\n                    wave = trim_silence(wave, teacher_rate)\n                    wave = resample_poly(wave, SAMPLE_RATE // divisor, teacher_rate // divisor)\n                    seconds = len(wave) / SAMPLE_RATE\n                    if not 1.0 <= seconds <= 12:\n                        skipped += 1\n                        continue\n                    rms = np.sqrt(np.mean(wave ** 2)) + 1e-8\n                    wave = np.clip(wave * (0.1 / rms), -0.99, 0.99)  # consistent loudness\n                    clip_id = f"{args.split}-{count:05d}"\n                    relative = f"wavs/{clip_id}.wav"\n                    sf.write(staging / relative, wave, SAMPLE_RATE, subtype="PCM_16")\n                    manifest.write(json.dumps({\n                        "id": clip_id, "audio": relative, "transcription": text, "phonemes": ipa, "phonemes_dataset": dataset_ipa,\n                        "source_split": args.split, "source_shard": shard, "source_row": index,\n                        "teacher": TEACHER, "teacher_revision": TEACHER_REVISION,\n                    }, ensure_ascii=False) + "\\n")\n                    count += 1\n                    if count % 500 == 0:\n                        print(f"{count} sentences synthesized", flush=True)\n                    if count == args.limit:\n                        break\n          except Exception as error:  # e.g. the corpus download failed for good\n            failure = error\n            print(f"Corpus reading stopped after {count} sentences: {error!r}", flush=True)\n        finally:\n            source.close()\n        if failure is not None and count < max(1, args.limit // 2):\n            raise failure\n        if count < max(1, args.limit // 2):\n            raise RuntimeError(f"Expected {args.limit} sentences and synthesized {count}")\n        if count < args.limit:\n            print(f"Warning: the split ran out at {count} of {args.limit} sentences; continuing.")\n        staging.rename(args.output)\n    print(f"Synthesized {count} teacher clips ({skipped} skipped) to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n',
}
SOURCE_DIGEST = '271cc57e34dc27259d8b865d017d8bfabfba2f005b51c5cc28a2013a29f29422'
for name, content in SOURCES.items():
    (PROJECT / name).write_text(content, encoding='utf-8')

## Set up the runtime

Kaggle supplies PyTorch and TorchAudio. This cell checks that those builds can
run the CUDA training and CPU forced-alignment kernels, installs the remaining
dependencies, then verifies the original sanoTTS int8 fixtures.

In [ ]:
python("-m", "pip", "install", "--quiet", "transformers==4.57.6",
       "huggingface_hub>=0.34,<1", "onnxruntime", "pyarrow", "requests", "scipy", "soundfile")

import torch
import torchaudio
from packaging.version import Version

if Version(torch.__version__.split("+")[0]) < Version("2.6"):
    raise RuntimeError("Select a recent Kaggle GPU image with PyTorch 2.6 or later.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if REQUIRE_GPU and DEVICE != "cuda":
    raise RuntimeError("Enable a GPU in Kaggle Notebook Settings and restart the session.")
if DEVICE == "cuda":
    torch.zeros(1, device=DEVICE).add_(1).cpu()
    print(torch.cuda.get_device_name(0))
torchaudio.functional.forced_align(
    torch.zeros(1, 3, 2).log_softmax(-1), torch.tensor([[1]], dtype=torch.int32)
)
print(f"PyTorch {torch.__version__} | TorchAudio {torchaudio.__version__} | {DEVICE}")

config = {
    "source_digest": SOURCE_DIGEST, "device": DEVICE,
    "train_candidates": TRAIN_CANDIDATES, "eval_candidates": EVAL_CANDIDATES,
    "duration_steps": DURATION_STEPS, "acoustic_steps": ACOUSTIC_STEPS,
    "decoder_steps": DECODER_STEPS, "decoder_ft_steps": DECODER_FT_STEPS,
    "single_voice": SINGLE_VOICE, "teacher": TEACHER,
    "teacher_noise_scale": TEACHER_NOISE_SCALE, "acoustic_adv_weight": ACOUSTIC_ADV_WEIGHT,
    "acoustic_mse_weight": ACOUSTIC_MSE_WEIGHT, "min_alignment_confidence": MIN_ALIGNMENT_CONFIDENCE,
    "decoder_predicted_fraction": DECODER_PREDICTED_FRACTION,
    "decoder_adv_weight": DECODER_ADV_WEIGHT,
    "duration_hidden": DURATION_HIDDEN, "acoustic_hidden": ACOUSTIC_HIDDEN, "acoustic_depth": ACOUSTIC_DEPTH,
}
config_path = RUNS / "run-config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise RuntimeError("Set a new RUN_NAME when changing the run configuration.")
config_path.write_text(json.dumps(config, indent=2))
run("make", "check-upstream")
budget = subprocess.run([sys.executable, "audit_budget.py", "128", "--duration-hidden", str(DURATION_HIDDEN),
                         "--acoustic-hidden", str(ACOUSTIC_HIDDEN), "--acoustic-depth", str(ACOUSTIC_DEPTH)],
                        cwd=PROJECT, env=ENV, capture_output=True, text=True)
print(budget.stdout + budget.stderr)
if budget.returncode:
    raise RuntimeError("Model sizes are over the parameter budget")
PARAMETERS = int(re.search(r"= ([\d,]+) parameters", budget.stdout)[1].replace(",", ""))
if not shutil.which("espeak-ng"):
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-y", "-qq", "espeak-ng")

## Build, align, and pack the training audio

With `TEACHER = True`, `teacher_data.py` streams UrduTTS sentences, expands numbers,
phonemizes them with eSpeak NG (the same front end used for synthesis), and has
MMS-TTS Urdu read each one on the GPU. With `TEACHER = False` it uses the human
recordings and keeps one pitch cluster, as in v3. Wav2Vec2 then aligns phones on
the GPU, and silences between words become `|` pause tokens. The train split
defines the shared phone map.


In [ ]:
VOCAB = CACHE / "phones.json"
VOICE_FILTER = CACHE / "train" / "utterances-voice.json"

for split, count in (("train", TRAIN_CANDIDATES), ("eval", EVAL_CANDIDATES)):
    audio = CACHE / split
    raw = audio / "utterances.jsonl"
    aligned = audio / "aligned.jsonl"
    pack = CACHE / f"{split}-pack"
    if not raw.is_file():
        if TEACHER:
            python("teacher_data.py", "--split", split, "--limit", count, "--output", audio,
                   "--device", DEVICE, "--noise-scale", TEACHER_NOISE_SCALE)
        else:
            python("prepare_data.py", "--split", split, "--limit", count, "--output", audio)
    manifest = raw
    if SINGLE_VOICE and not TEACHER:
        manifest = audio / "utterances-voice.jsonl"
        if not manifest.is_file():
            center = ["--center", json.loads(VOICE_FILTER.read_text())["center_hz"]] if split == "eval" else []
            python("speaker_filter.py", raw, manifest, *center, "--min-clips", 16 if SMOKE_TEST or split == "eval" else 200)
    if not aligned.is_file():
        python("align_data.py", manifest, aligned, VOCAB, "--device", DEVICE,
               "--min-confidence", MIN_ALIGNMENT_CONFIDENCE)
    if not (pack / "rows.json").is_file():
        python("build_pack.py", aligned, VOCAB, pack)
    rows = json.loads((pack / "rows.json").read_text())
    print(f"{split}: {len(rows)} accepted from {count} candidates")

TRAIN = CACHE / "train-pack"
EVAL = CACHE / "eval-pack"


## Listen to the teacher

These are training targets. The finished small model will at best sound like this
voice. If you do not like it, stop here and set `TEACHER = False`.


In [ ]:
from IPython.display import Audio, display

for line in (CACHE / "train/aligned.jsonl").read_text(encoding="utf-8").splitlines()[:3]:
    row = json.loads(line)
    print(row["transcription"])
    print(" ".join(row["phones"]))
    display(Audio(filename=row["audio"]))


## Train duration, acoustics, and waveform decoder

The decoder starts from the published `heart-nano` weights and trains in two
stages, both with GAN discriminators (multi-period and multi-resolution) plus
spectral, phase and feature-matching losses. Stage 1 uses crops of the true mel.
Stage 2 continues, with the same discriminators, on 80% acoustic-model predictions
and 20% true mels, with int8 rounding simulated, so the decoder learns to render
what it will actually receive. The two stages share the time left before
`TIME_LIMIT_HOURS`.
The acoustic model uses sharpening losses so it does not average pitch harmonics
into a blur. Each completed stage can be reused when rerunning cells in the same
session.


In [ ]:
TOOLS = PROJECT / "vendor/sanoTTS/tools"
DURATION = RUNS / "duration/duration-student.pt"
ACOUSTIC = RUNS / "acoustic/latent-student.pt"
DECODER_GT = RUNS / "decoder-stage1.pt"
DECODER = RUNS / "decoder.pt"

if not (DURATION.parent / "train-report.json").is_file() or not DURATION.is_file():
    python(TOOLS / "train_roota_piper_duration_student.py", "--pack-dir", TRAIN,
           "--eval-pack-dir", EVAL, "--out-dir", DURATION.parent,
           "--hidden", DURATION_HIDDEN, "--depth", 3, "--kernel-size", 5, "--duration-vocab-size", 128,
           "--max-duration", 80, "--pause-token-ids", "", "--steps", DURATION_STEPS,
           "--device", DEVICE)
if not (ACOUSTIC.parent / "train-report.json").is_file() or not ACOUSTIC.is_file():
    adversarial = ["--latent-adv-weight", ACOUSTIC_ADV_WEIGHT,
                   "--latent-adv-start-step", max(1, ACOUSTIC_STEPS // 4)] if ACOUSTIC_ADV_WEIGHT else []
    python(TOOLS / "train_roota_piper_latent_student.py", "--pack-dir", TRAIN,
           "--eval-pack-dir", EVAL, "--out-dir", ACOUSTIC.parent,
           "--architecture", "token_context", "--hidden", ACOUSTIC_HIDDEN, "--depth", ACOUSTIC_DEPTH,
           "--token-depth", ACOUSTIC_DEPTH, "--kernel-size", 5, "--vocab-size", 128,
           "--skip-render", "--steps", ACOUSTIC_STEPS, "--device", DEVICE,
           "--lr", 1e-3, "--mse-weight", ACOUSTIC_MSE_WEIGHT,
           "--checkpoint-steps", CHECKPOINT_STEPS, "--norm-l1-weight", 0.5, "--delta-l1-weight", 0.5,
           "--channel-stat-weight", 0.25, *adversarial)

def minutes_left(share):
    """Share of the time left before TIME_LIMIT_HOURS, keeping 40 min for export and zip."""
    left = TIME_LIMIT_HOURS * 60 - (time.time() - RUN_START) / 60 - 40
    print(f"{left:.0f} min left for decoder training; this stage gets {max(10, left * share):.0f} min")
    return max(10, left * share)

def train_decoder(target, *extra, steps, share):
    if target.is_file():
        return
    with tempfile.TemporaryDirectory(dir=RUNS) as temp:
        pending = Path(temp) / target.name
        python("decoder.py", CACHE / "train/aligned.jsonl", TRAIN, pending,
               "--steps", steps, "--device", DEVICE, "--adv-weight", DECODER_ADV_WEIGHT,
               "--max-minutes", round(minutes_left(share), 1), *extra)
        pending.replace(target)

train_decoder(DECODER_GT, steps=DECODER_STEPS, share=0.55)
train_decoder(DECODER, "--init", DECODER_GT, "--acoustic-checkpoint", ACOUSTIC, "--qat",
              "--predicted-fraction", DECODER_PREDICTED_FRACTION, "--lr", 1e-4, "--min-lr", 1e-5, steps=DECODER_FT_STEPS, share=1.0)


## Export and compare the int8 voice

The parity check compares the upstream C runtime with floating-point model
output on held-out rows. It measures export fidelity. Listen to the Urdu
script test below to judge speech quality.
A failed parity check is recorded in the bundle so the PyTorch checkpoints
remain available for inspection and retraining.

In [ ]:
VOICE = RUNS / "int8"
if not VOICE.exists():
    with tempfile.TemporaryDirectory(dir=RUNS) as temp:
        pending = Path(temp) / "int8"
        python("export_nano.py", "--base", PROJECT, "--duration-checkpoint", DURATION,
               "--acoustic-checkpoint", ACOUSTIC, "--decoder-checkpoint", DECODER,
               "--pack", EVAL, "--vocabulary", VOCAB, "--out", pending)
        pending.rename(VOICE)

PARITY_LOG = RUNS / "int8-parity.txt"
with PARITY_LOG.open("w") as log:
    parity = run("make", "check-voice", f"VOICE={VOICE}", check=False, stdout=log)
print(PARITY_LOG.read_text())
PARITY_PASSED = parity.returncode == 0
print("Int8 parity:", "PASS" if PARITY_PASSED else "FAIL")

## Test Urdu script and numbers

Edit `URDU_TEXT` below and rerun just this cell. Enter Urdu script, including
text copied from a Nastaliq source. Urdu, Arabic, and Western digits all work.
Whole numbers use Urdu counting words, decimals read each fractional digit,
and leading-zero numbers read digit by digit. Set `READ_ALL_DIGITS = True` for
codes without leading zeros. Write dates, times, ranges, and fractions out in
words. Try `قیمت ۱٬۲۵۰ روپے ہے۔`, `شرح ۳٫۰۵٪ ہے۔`, or `کوڈ ۰۰۷ ہے۔`.

The cell displays the expanded Urdu and IPA. If the pilot's phone inventory
cannot represent a pronunciation, try another sentence or train on more varied
clips.

In [ ]:
from IPython.display import Audio, display

URDU_TEXT = "قیمت ۱٬۲۵۰ روپے ہے۔"
READ_ALL_DIGITS = False

URDU_SAMPLE = RUNS / "urdu-script-example.wav"
with tempfile.TemporaryDirectory(dir=RUNS) as temp:
    pending = Path(temp) / URDU_SAMPLE.name
    options = ["--digits"] if READ_ALL_DIGITS else []
    result = python("urdu_frontend.py", VOICE, URDU_TEXT, pending, *options, check=False)
    if result.returncode == 0:
        pending.replace(URDU_SAMPLE)
        display(Audio(filename=str(URDU_SAMPLE)))
    else:
        print("Urdu text test failed. See the message above and try a shorter or simpler input.")

## Compare ground truth with each stage

This renders held-out recordings through the model one stage at a time. Listen to
the four files for each clip, left to right:

| File | What it tests | If it sounds like a chorus |
|---|---|---|
| 1 ground truth | the training target (teacher audio in v4) | n/a |
| 2 decoder only | true mel → decoder | the **decoder** is the problem |
| 3 acoustic | predicted mel → decoder | the **acoustic model** is the problem |
| 4 int8 runtime | phonemes → shipped C runtime | **quantization** or predicted durations |

The mel image shows the true and predicted spectrogram side by side. Clear
horizontal harmonic stripes in the true one that are smeared in the predicted one
mean the acoustic model is blurring pitch. Median pitch per file is in the table:
a stage whose pitch drifts from the ground truth is where it goes wrong.


In [ ]:
from IPython.display import Audio, Image, display

COMPARE = RUNS / "compare"
if COMPARE.exists():
    shutil.rmtree(COMPARE)
python("compare_voice.py", CACHE / "eval/aligned.jsonl", EVAL, "--decoder", DECODER,
       "--acoustic", ACOUSTIC, "--voice", VOICE, "--out", COMPARE, "--count", 4,
       "--device", DEVICE, stdout=subprocess.DEVNULL)
report = json.loads((COMPARE / "report.json").read_text())
for item in report:
    print(f"\n=== {item['row']}  |  mel L1 {item['mel_l1']:.2f}  cosine {item['mel_cosine']:.3f}")
    print(item["text"])
    for name, path in item["files"].items():
        variation = item["pitch_variation_semitones"][name]
        print(name, f"(pitch variation {variation if variation is not None else float('nan'):.1f} semitones; "
              "robotic = low)")
        display(Audio(filename=path))
if (COMPARE / "mels.png").is_file():
    display(Image(filename=str(COMPARE / "mels.png")))


## Download the model bundle

The ZIP includes int8 weights, the phone map, the runtime parity log, training
reports, PyTorch checkpoints, and the generated sample when rendering succeeds.
The corpus recordings and training-only aligner stay outside the archive.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

COMPARE = RUNS / "compare"
status = RUNS / "run-status.json"
status.write_text(json.dumps({
    "parameters": PARAMETERS,
    "sanoTTS_int8_parity_passed": PARITY_PASSED,
    "phoneme_input": True,
    "teacher": "facebook/mms-tts-urd-script_arabic" if TEACHER else None,
    "word_boundary_token": "|",
    "notebook_text_input": "Urdu script and numbers via eSpeak NG",
    "dataset": "kaab4321/UrduTTS",
    "corpus_license": "CC BY-NC 4.0",
}, indent=2))

archive = WORKING / f"nano-urdu-{RUN_NAME}.zip"
with tempfile.TemporaryDirectory(dir=WORKING) as temp:
    pending = Path(temp) / archive.name
    with ZipFile(pending, "w", compression=ZIP_DEFLATED) as bundle:
        for path in sorted(VOICE.iterdir()):
            if path.is_file():
                bundle.write(path, f"int8/{path.name}")
        for path in (DURATION, ACOUSTIC, DECODER, PARITY_LOG, status, config_path,
                     DURATION.parent / "train-report.json", ACOUSTIC.parent / "train-report.json",
                     DECODER_GT, URDU_SAMPLE, *sorted(ACOUSTIC.parent.glob("checkpoint_step*.pt")), *sorted(COMPARE.rglob("*.*"))):
            if path.is_file():
                bundle.write(path, path.relative_to(RUNS).as_posix())
        for split in ("train", "eval"):
            skips = CACHE / split / "aligned-skips.json"
            if skips.is_file():
                bundle.write(skips, f"alignment-skips-{split}.json")
    pending.replace(archive)
with ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print(f"{archive} ({archive.stat().st_size:,} bytes)")